In [1]:
!pip install torchmetrics[image]

In [2]:
import os
import random
from math import log2
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
from torchmetrics import StructuralSimilarityIndexMeasure

# ==========================================
# CONFIGURATION & HYPERPARAMETERS
# ==========================================
START_TRAIN_AT_IMG_SIZE = 4 
TARGET_IMG_SIZE = 1024 
DATASET = '/kaggle/input/datasets/lavender991/ned-new/reduced_kidney' # <--- Verify this path!
CHECKPOINT_GEN = "/kaggle/working/g_sa.pth"
CHECKPOINT_CRITIC = "/kaggle/working/c_sa.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_MODEL = True
LOAD_MODEL = False
LEARNING_RATE = 1e-3

# Optimized for 2x Kaggle T4 GPUs 
BATCH_SIZES = [32, 32, 32, 16, 16, 16, 8, 4, 2] 
# Staggered epochs (Total = 110)
PROGRESSIVE_EPOCHS = [1] * len(BATCH_SIZES)

CHANNELS_IMG = 3
Z_DIM = 100
IN_CHANNELS = 256
LAMBDA_GP = 10
NUM_WORKERS = 4

torch.backends.cudnn.benchmarks = True

def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
factors = [1, 1, 1, 1, 1 / 2, 1 / 4, 1 / 8, 1 / 16, 1 / 32]

class WSConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, gain=2):
        super(WSConv2d, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.scale = (gain / (in_channels * (kernel_size ** 2))) ** 0.5
        self.bias = self.conv.bias
        self.conv.bias = None
        nn.init.normal_(self.conv.weight)
        nn.init.zeros_(self.bias)

    def forward(self, x):
        return self.conv(x * self.scale) + self.bias.view(1, self.bias.shape[0], 1, 1)

class PixelNorm(nn.Module):
    def __init__(self):
        super(PixelNorm, self).__init__()
        self.epsilon = 1e-8

    def forward(self, x):
        return x / torch.sqrt(torch.mean(x ** 2, dim=1, keepdim=True) + self.epsilon)

class SelfAttention(nn.Module):
    """ NOVELTY: Self-attention Layer for preserving global anatomical structure """
    def __init__(self, in_channels):
        super(SelfAttention, self).__init__()
        self.query_conv = nn.Conv2d(in_channels, in_channels // 8, kernel_size=1)
        self.key_conv = nn.Conv2d(in_channels, in_channels // 8, kernel_size=1)
        self.value_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        batch_size, C, width, height = x.size()
        proj_query = self.query_conv(x).view(batch_size, -1, width * height).permute(0, 2, 1)
        proj_key = self.key_conv(x).view(batch_size, -1, width * height)
        energy = torch.bmm(proj_query, proj_key)
        attention = self.softmax(energy)
        proj_value = self.value_conv(x).view(batch_size, -1, width * height)
        
        out = torch.bmm(proj_value, attention.permute(0, 2, 1))
        out = out.view(batch_size, C, width, height)
        out = self.gamma * out + x
        return out

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, use_pixelnorm=True):
        super(ConvBlock, self).__init__()
        self.use_pn = use_pixelnorm
        self.conv1 = WSConv2d(in_channels, out_channels)
        self.conv2 = WSConv2d(out_channels, out_channels)
        self.leaky = nn.LeakyReLU(0.2)
        self.pn = PixelNorm()

    def forward(self, x):
        x = self.leaky(self.conv1(x))
        x = self.pn(x) if self.use_pn else x
        x = self.leaky(self.conv2(x))
        x = self.pn(x) if self.use_pn else x
        return x

class Generator(nn.Module):
    def __init__(self, z_dim, in_channels, img_channels=3):
        super(Generator, self).__init__()
        self.initial = nn.Sequential(
            PixelNorm(),
            nn.ConvTranspose2d(z_dim, in_channels, 4, 1, 0),
            nn.LeakyReLU(0.2),
            WSConv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2),
            PixelNorm(),
        )
        self.initial_rgb = WSConv2d(in_channels, img_channels, kernel_size=1, stride=1, padding=0)
        self.prog_blocks, self.rgb_layers = nn.ModuleList([]), nn.ModuleList([self.initial_rgb])
        
        # Self-Attention initialized for the 64x64 stage
        attention_channels = int(in_channels * factors[4])
        self.attention = SelfAttention(attention_channels)

        for i in range(len(factors) - 1):
            conv_in_c = int(in_channels * factors[i])
            conv_out_c = int(in_channels * factors[i + 1])
            self.prog_blocks.append(ConvBlock(conv_in_c, conv_out_c))
            self.rgb_layers.append(WSConv2d(conv_out_c, img_channels, kernel_size=1, stride=1, padding=0))

    def fade_in(self, alpha, upscaled, generated):
        return torch.tanh(alpha * generated + (1 - alpha) * upscaled)

    def forward(self, x, alpha, steps):
        out = self.initial(x)
        if steps == 0:
            return self.initial_rgb(out)
            
        for step in range(steps):
            upscaled = F.interpolate(out, scale_factor=2, mode="nearest")
            out = self.prog_blocks[step](upscaled)
            
            # Apply attention at 64x64 output
            if step == 3: 
                out = self.attention(out)

        final_upscaled = self.rgb_layers[steps - 1](upscaled)
        final_out = self.rgb_layers[steps](out)
        return self.fade_in(alpha, final_upscaled, final_out)

class Discriminator(nn.Module):
    def __init__(self, z_dim, in_channels, img_channels=3):
        super(Discriminator, self).__init__()
        self.prog_blocks, self.rgb_layers = nn.ModuleList([]), nn.ModuleList([])
        self.leaky = nn.LeakyReLU(0.2)
        
        attention_channels = int(in_channels * factors[4])
        self.attention = SelfAttention(attention_channels)

        for i in range(len(factors) - 1, 0, -1):
            conv_in = int(in_channels * factors[i])
            conv_out = int(in_channels * factors[i - 1])
            self.prog_blocks.append(ConvBlock(conv_in, conv_out, use_pixelnorm=False))
            self.rgb_layers.append(WSConv2d(img_channels, conv_in, kernel_size=1, stride=1, padding=0))

        self.initial_rgb = WSConv2d(img_channels, in_channels, kernel_size=1, stride=1, padding=0)
        self.rgb_layers.append(self.initial_rgb)
        self.avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

        self.final_block = nn.Sequential(
            WSConv2d(in_channels + 1, in_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
            WSConv2d(in_channels, in_channels, kernel_size=4, padding=0, stride=1),
            nn.LeakyReLU(0.2),
            WSConv2d(in_channels, 1, kernel_size=1, padding=0, stride=1),
        )

    def fade_in(self, alpha, downscaled, out):
        return alpha * out + (1 - alpha) * downscaled

    def minibatch_std(self, x):
        # FIX: Added unbiased=False
        batch_statistics = torch.std(x, dim=0, unbiased=False).mean().repeat(x.shape[0], 1, x.shape[2], x.shape[3])
        return torch.cat([x, batch_statistics], dim=1)

    def forward(self, x, alpha, steps):
        cur_step = len(self.prog_blocks) - steps
        out = self.leaky(self.rgb_layers[cur_step](x))

        if steps == 0:
            out = self.minibatch_std(out)
            return self.final_block(out).view(out.shape[0], -1)

        # Apply attention to 64x64 input
        if cur_step == 4:
            out = self.attention(out)

        downscaled = self.leaky(self.rgb_layers[cur_step + 1](self.avg_pool(x)))
        out = self.avg_pool(self.prog_blocks[cur_step](out))
        out = self.fade_in(alpha, downscaled, out)

        for step in range(cur_step + 1, len(self.prog_blocks)):
            if step == 4:
                out = self.attention(out)
            out = self.prog_blocks[step](out)
            out = self.avg_pool(out)

        out = self.minibatch_std(out)
        return self.final_block(out).view(out.shape[0], -1)

In [4]:
def get_loader(image_size):
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.Normalize([0.5]*CHANNELS_IMG, [0.5]*CHANNELS_IMG),
    ])
    batch_size = BATCH_SIZES[int(log2(image_size / 4))]
    dataset = datasets.ImageFolder(root=DATASET, transform=transform)
    
    # FIX: Added drop_last=True
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True, 
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True 
    )
    return loader, dataset

def gradient_penalty(critic, real, fake, alpha, train_step, device="cpu"):
    BATCH_SIZE, C, H, W = real.shape
    beta = torch.rand((BATCH_SIZE, 1, 1, 1)).repeat(1, C, H, W).to(device)
    interpolated_images = real * beta + fake.detach() * (1 - beta)
    interpolated_images.requires_grad_(True)

    mixed_scores = critic(interpolated_images, alpha, train_step)
    gradient = torch.autograd.grad(
        inputs=interpolated_images, outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True, retain_graph=True,
    )[0]
    gradient = gradient.view(gradient.shape[0], -1)
    gradient_norm = gradient.norm(2, dim=1)
    return torch.mean((gradient_norm - 1) ** 2)

def save_checkpoint(model, optimizer, filename):
    model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    checkpoint = {"state_dict": model_state, "optimizer": optimizer.state_dict()}
    torch.save(checkpoint, filename)

def load_checkpoint(checkpoint_file, model, optimizer, lr):
    checkpoint = torch.load(checkpoint_file, map_location=DEVICE)
    model_state = model.module if isinstance(model, nn.DataParallel) else model
    model_state.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

In [5]:
def train_fn(critic, gen, loader, dataset, step, alpha, opt_critic, opt_gen, scaler_gen, scaler_critic):
    loop = tqdm(loader, leave=True)
    for batch_idx, (real, _) in enumerate(loop):
        real = real.to(DEVICE)
        cur_batch_size = real.shape[0]
        noise = torch.randn(cur_batch_size, Z_DIM, 1, 1).to(DEVICE)

        # ---------------------
        # Train Critic
        # ---------------------
        with torch.amp.autocast('cuda'):
            fake = gen(noise, alpha, step)
            critic_real = critic(real, alpha, step)
            # .detach() is CORRECT here because we don't backprop to the generator yet
            critic_fake = critic(fake.detach(), alpha, step) 
            
        # GP calculated outside autocast in FP32 to prevent NaNs
        gp = gradient_penalty(critic, real, fake, alpha, step, device=DEVICE)

        with torch.amp.autocast('cuda'):
            loss_critic = (
                -(torch.mean(critic_real) - torch.mean(critic_fake))
                + LAMBDA_GP * gp
                + (0.001 * torch.mean(critic_real ** 2))
            )

        opt_critic.zero_grad()
        scaler_critic.scale(loss_critic).backward()
        scaler_critic.step(opt_critic)
        scaler_critic.update()

        # ---------------------
        # Train Generator
        # ---------------------
        with torch.amp.autocast('cuda'):
            # THE FIX: Removed .detach() here so gradients flow back to the Generator
            gen_fake = critic(fake, alpha, step) 
            loss_gen = -torch.mean(gen_fake)

        opt_gen.zero_grad()
        scaler_gen.scale(loss_gen).backward()
        scaler_gen.step(opt_gen)
        scaler_gen.update()

        # ---------------------
        # Update Alpha
        # ---------------------
        alpha += cur_batch_size / ((PROGRESSIVE_EPOCHS[step] * 0.5) * len(dataset))
        alpha = min(alpha, 1)

        loop.set_postfix(gp=gp.item(), loss_critic=loss_critic.item())

    return alpha

In [6]:
def calculate_metrics(real_loader, num_images=64, target_size=1024):
    print(f"\n--- Calculating  Metrics for {num_images} images at {target_size}x{target_size} ---")
    gen = Generator(Z_DIM, IN_CHANNELS, img_channels=CHANNELS_IMG).to(DEVICE)
    if os.path.exists(CHECKPOINT_GEN):
        checkpoint = torch.load(CHECKPOINT_GEN, map_location=DEVICE)
        gen.load_state_dict(checkpoint["state_dict"])
    gen.eval()

    fid = FrechetInceptionDistance(feature=2048).to(DEVICE)
    inception = InceptionScore().to(DEVICE)
    ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(DEVICE)
    
    steps = int(log2(target_size / 4))
    images_processed = 0
    ssim_total = 0.0
    batches = 0
    
    with torch.no_grad():
        # Process in safe batches to prevent Dimension Mismatch & OOM crashes
        for real_batch, _ in real_loader:
            current_batch_size = real_batch.shape[0]
            if images_processed >= num_images:
                break
                
            real_batch = real_batch.to(DEVICE)
            noise = torch.randn(current_batch_size, Z_DIM, 1, 1).to(DEVICE)
            fake_batch = gen(noise, alpha=1.0, steps=steps)
            
            real_uint8 = ((real_batch * 0.5 + 0.5) * 255).to(torch.uint8)
            fake_uint8 = ((fake_batch * 0.5 + 0.5) * 255).to(torch.uint8)
            
            fid.update(real_uint8, real=True)
            fid.update(fake_uint8, real=False)
            inception.update(fake_uint8)
            
            real_norm = real_batch * 0.5 + 0.5
            fake_norm = fake_batch * 0.5 + 0.5
            ssim_total += ssim(fake_norm, real_norm).item()
            
            images_processed += current_batch_size
            batches += 1

    print(f"✅ FID Score: {fid.compute().item():.2f} (Lower is better)")
    print(f"✅ Inception Score: {inception.compute()[0].item():.2f} (Higher is better)")
    print(f"✅ SSIM: {ssim_total / batches:.4f} (Closer to 1.0 is better)")

In [ ]:
if __name__ == "__main__":
    seed_everything()
    
    gen = Generator(Z_DIM, IN_CHANNELS, img_channels=CHANNELS_IMG).to(DEVICE)
    critic = Discriminator(Z_DIM, IN_CHANNELS, img_channels=CHANNELS_IMG).to(DEVICE)

    if torch.cuda.device_count() > 1:
        print(f"🚀 Utilizing {torch.cuda.device_count()} GPUs for SA-MedGAN!")
        gen = nn.DataParallel(gen)
        critic = nn.DataParallel(critic)

    opt_gen = optim.Adam(gen.parameters(), lr=LEARNING_RATE, betas=(0.0, 0.99))
    opt_critic = optim.Adam(critic.parameters(), lr=LEARNING_RATE, betas=(0.0, 0.99))
    
    scaler_critic = torch.amp.GradScaler('cuda')
    scaler_gen = torch.amp.GradScaler('cuda')
    
    if LOAD_MODEL and os.path.exists(CHECKPOINT_GEN) and os.path.exists(CHECKPOINT_CRITIC):
        load_checkpoint(CHECKPOINT_GEN, gen, opt_gen, LEARNING_RATE)
        load_checkpoint(CHECKPOINT_CRITIC, critic, opt_critic, LEARNING_RATE)

    gen.train()
    critic.train()

    step = int(log2(START_TRAIN_AT_IMG_SIZE / 4))
    end_step = int(log2(TARGET_IMG_SIZE / 4)) 

    # 1. Start Training Loop
    for num_epochs in PROGRESSIVE_EPOCHS[step:end_step + 1]:
        alpha = 1e-5
        image_size = 4 * 2 ** step
        loader, dataset = get_loader(image_size)
        print(f"\n--- Training Phase: {image_size}x{image_size} ---")

        for epoch in range(num_epochs):
            print(f"Epoch [{epoch+1}/{num_epochs}]")
            alpha = train_fn(critic, gen, loader, dataset, step, alpha, opt_critic, opt_gen, scaler_gen, scaler_critic)

            if SAVE_MODEL:
                save_checkpoint(gen, opt_gen, filename=CHECKPOINT_GEN)
                save_checkpoint(critic, opt_critic, filename=CHECKPOINT_CRITIC)
                
        step += 1 
        
    loader, _ = get_loader(TARGET_IMG_SIZE)
    calculate_metrics(loader, num_images=64, target_size=TARGET_IMG_SIZE)

🚀 Utilizing 2 GPUs for SA-MedGAN!

--- Training Phase: 4x4 ---
Epoch [1/1]


100%|██████████| 7/7 [00:03<00:00,  1.99it/s, gp=0.0147, loss_critic=-0.454] 



--- Training Phase: 8x8 ---
Epoch [1/1]


100%|██████████| 7/7 [00:01<00:00,  3.78it/s, gp=0.00963, loss_critic=-1.76]



--- Training Phase: 16x16 ---
Epoch [1/1]


  0%|          | 0/7 [00:00<?, ?it/s]